# Perfilado de datasets — Proyecto RecSys (H3)

```
profile_datasets — Perfilado de datasets para Proyecto RecSys (H3)
==================================================================
Imprime esquema (columnas/tipos), conteos (usuarios/ítems/interacciones),
faltantes y solapamientos de cada dataset, para confirmar DATASETS.md.

Cómo correr en Google Colab:
  1) Ejecuta las celdas de arriba a abajo (cada dataset es una celda → si
     una falla o llena la RAM, reinicias el runtime y sigues con la siguiente).
  2) Para Kaggle: sube tu kaggle.json o usa kagglehub.login().
  3) Pega de vuelta la salida y se actualiza DATASETS.md.
```

## Setup (instalación + helpers)

In [1]:
# !pip -q install kagglehub pandas gdown

import gzip, ast, io, json, os, urllib.request
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

HEADERS = {"User-Agent": "Mozilla/5.0"}


def banner(txt):
    print("\n" + "=" * 78 + f"\n{txt}\n" + "=" * 78)


def safe_nunique(s):
    """nunique tolerante a columnas con listas/dicts (unhashable)."""
    try:
        return int(s.nunique(dropna=True))
    except TypeError:
        return int(s.astype(str).nunique(dropna=True))


def profile_df(df, name, id_cols=()):
    """Imprime shape, columnas+tipos, %faltantes, nunique y head(3)."""
    print(f"\n--- {name} ---")
    print(f"shape: {df.shape[0]:,} filas x {df.shape[1]} columnas")
    info = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "%_missing": (df.isna().mean() * 100).round(2),
    })
    info["n_unique"] = [safe_nunique(df[c]) for c in df.columns]
    print(info.to_string())
    for c in id_cols:
        if c in df.columns:
            print(f"  · {c}: {safe_nunique(df[c]):,} valores únicos")
    print("head(3):")
    print(df.head(3).to_string())

## KOZYRIEV — Game Recommendations on Steam

In [2]:
banner("DATASET 1 · KOZYRIEV (antonkozyriev/game-recommendations-on-steam)")
try:
    import kagglehub
    kpath = kagglehub.dataset_download("antonkozyriev/game-recommendations-on-steam")
    print("Descargado en:", kpath, "\nArchivos:", os.listdir(kpath))

    games = pd.read_csv(os.path.join(kpath, "games.csv"))
    users = pd.read_csv(os.path.join(kpath, "users.csv"))
    profile_df(games, "games.csv", id_cols=["app_id"])
    profile_df(users, "users.csv", id_cols=["user_id"])

    # games_metadata.json (1 objeto JSON por línea; 'tags' es una lista)
    meta_rows = []
    with open(os.path.join(kpath, "games_metadata.json"), encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                meta_rows.append(json.loads(line))
    metadata = pd.DataFrame(meta_rows)
    profile_df(metadata, "games_metadata.json", id_cols=["app_id"])
    if "tags" in metadata.columns:
        print("ejemplo tags[0]:", metadata["tags"].iloc[0])

    # recommendations.csv (~41M filas): muestra para perfilar + conteo total
    recs_head = pd.read_csv(os.path.join(kpath, "recommendations.csv"), nrows=200_000)
    profile_df(recs_head, "recommendations.csv (muestra 200k)",
               id_cols=["app_id", "user_id"])
    n_recs = sum(1 for _ in open(os.path.join(kpath, "recommendations.csv"),
                                 encoding="utf-8")) - 1
    print(f"\nrecommendations.csv TOTAL filas: {n_recs:,}")
    kozyriev_games = set(games["app_id"])
except Exception as e:
    print("‼ Kozyriev falló:", repr(e))
    kozyriev_games = set()


DATASET 1 · KOZYRIEV (antonkozyriev/game-recommendations-on-steam)
Using Colab cache for faster access to the 'game-recommendations-on-steam' dataset.
Descargado en: /kaggle/input/game-recommendations-on-steam 
Archivos: ['games_metadata.json', 'users.csv', 'games.csv', 'recommendations.csv']

--- games.csv ---
shape: 50,872 filas x 13 columnas
                  dtype  n_missing  %_missing  n_unique
app_id            int64          0        0.0     50872
title            object          0        0.0     50751
date_release     object          0        0.0      4292
win                bool          0        0.0         2
mac                bool          0        0.0         2
linux              bool          0        0.0         2
rating           object          0        0.0         9
positive_ratio    int64          0        0.0       100
user_reviews      int64          0        0.0      4847
price_final     float64          0        0.0       436
price_original  float64          0  

## FRONKONGAMES — Steam Games Dataset (aumento de Kozyriev)

In [3]:
banner("DATASET 1b · FRONKONGAMES (fronkongames/steam-games-dataset)")
try:
    import kagglehub
    fpath = kagglehub.dataset_download("fronkongames/steam-games-dataset")
    print("Descargado en:", fpath, "\nArchivos:", os.listdir(fpath))

    fcsv = [x for x in os.listdir(fpath) if x.endswith(".csv")]
    if fcsv:
        fronkon = pd.read_csv(os.path.join(fpath, fcsv[0]))
    else:
        fjson = [x for x in os.listdir(fpath) if x.endswith(".json")][0]
        fronkon = pd.read_json(os.path.join(fpath, fjson)).T

    print("\n>>> TODAS las columnas de FronkonGames:")
    print(list(fronkon.columns))
    profile_df(fronkon, "FronkonGames",
               id_cols=[c for c in ["AppID", "app_id"] if c in fronkon.columns])

    for col in ["Genres", "Developers", "Publishers", "Categories", "Tags"]:
        if col in fronkon.columns:
            print(f"\nEjemplos de '{col}':")
            print(fronkon[col].dropna().head(3).to_string())

    # >>> COBERTURA DEL JOIN: ¿cuántos juegos de Kozyriev reciben metadata?
    # (AppID viene como object y puede traer NaN/inf por filas corridas del CSV;
    #  between() descarta ambos antes del cast a int)
    idcol = "AppID" if "AppID" in fronkon.columns else (
        "app_id" if "app_id" in fronkon.columns else None)
    if idcol and kozyriev_games:
        ids = pd.to_numeric(fronkon[idcol], errors="coerce")
        fronkon_ids = set(ids[ids.between(1, 10_000_000)].astype("int64"))
        inter = kozyriev_games & fronkon_ids
        print(f"\n>>> COBERTURA JOIN Kozyriev∩FronkonGames por {idcol}:")
        print(f"    Juegos Kozyriev:     {len(kozyriev_games):,}")
        print(f"    Juegos FronkonGames: {len(fronkon_ids):,}")
        print(f"    Con metadata:        {len(inter):,} "
              f"({100*len(inter)/max(1,len(kozyriev_games)):.1f}% de Kozyriev)")
except Exception as e:
    print("‼ FronkonGames falló:", repr(e))


DATASET 1b · FRONKONGAMES (fronkongames/steam-games-dataset)
Using Colab cache for faster access to the 'steam-games-dataset' dataset.
Descargado en: /kaggle/input/steam-games-dataset 
Archivos: ['games.csv', 'games.json']

>>> TODAS las columnas de FronkonGames:
['AppID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 'Required age', 'Price', 'DiscountDLC count', 'About the game', 'Supported languages', 'Full audio languages', 'Reviews', 'Header image', 'Website', 'Support url', 'Support email', 'Windows', 'Mac', 'Linux', 'Metacritic score', 'Metacritic url', 'User score', 'Positive', 'Negative', 'Score rank', 'Achievements', 'Recommendations', 'Notes', 'Average playtime forever', 'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks', 'Developers', 'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots', 'Movies']

--- FronkonGames ---
shape: 125,855 filas x 39 columnas
                              dtype  n_missing  %_missing  n_unique
App

## UCSD / McAuley — Steam (Australian)   [2º dataset principal]

In [4]:
banner("DATASET 2 · UCSD/McAuley STEAM")
UCSD_URLS = {
    "steam_games":  "https://cseweb.ucsd.edu/~wckang/steam_games.json.gz",
    "users_items":  "https://mcauleylab.ucsd.edu/public_datasets/data/steam/australian_users_items.json.gz",
    "user_reviews": "https://mcauleylab.ucsd.edu/public_datasets/data/steam/australian_user_reviews.json.gz",
}


def load_ucsd(url, limit=None):
    """Los .json.gz de McAuley son dicts de Python por línea -> literal_eval.
    Se hace streaming (sin cargar el .gz completo en RAM)."""
    print("descargando", url)
    req = urllib.request.Request(url, headers=HEADERS)
    rows = []
    with urllib.request.urlopen(req) as resp, gzip.GzipFile(fileobj=resp) as gz:
        for i, line in enumerate(gz):
            rows.append(ast.literal_eval(line.decode("utf-8")))
            if limit and i + 1 >= limit:
                break
    return rows


try:
    games_u = pd.DataFrame(load_ucsd(UCSD_URLS["steam_games"]))
    print("\n>>> columnas steam_games:", list(games_u.columns))
    profile_df(games_u, "steam_games.json.gz",
               id_cols=[c for c in ["id", "app_name"] if c in games_u.columns])

    ui = load_ucsd(UCSD_URLS["users_items"])
    n_users_u = len(ui)
    n_inter_u = sum(u.get("items_count", len(u.get("items", []))) for u in ui)
    item_fields = list(ui[0]["items"][0].keys()) if ui and ui[0].get("items") else "N/A"
    print(f"\naustralian_users_items: {n_users_u:,} usuarios, "
          f"~{n_inter_u:,} interacciones (suma items_count)")
    print("campos por item:", item_fields)
except Exception as e:
    print("‼ UCSD falló:", repr(e))


DATASET 2 · UCSD/McAuley STEAM
descargando https://cseweb.ucsd.edu/~wckang/steam_games.json.gz

>>> columnas steam_games: ['publisher', 'genres', 'app_name', 'title', 'url', 'release_date', 'tags', 'discount_price', 'reviews_url', 'specs', 'price', 'early_access', 'id', 'developer', 'sentiment', 'metascore']

--- steam_games.json.gz ---
shape: 32,135 filas x 16 columnas
                  dtype  n_missing  %_missing  n_unique
publisher        object       8052      25.06      8239
genres           object       3283      10.22       884
app_name         object          2       0.01     32094
title            object       2050       6.38     30054
url              object          0       0.00     32135
release_date     object       2067       6.43      3582
tags             object        163       0.51     15396
discount_price  float64      31910      99.30        82
reviews_url      object          2       0.01     32132
specs            object        670       2.08      4650
price     

## SCGRec — dataset del paper (Yang et al. WWW'22, Google Drive)

In [8]:
banner("DATASET 2b · SCGRec (Yang et al. WWW'22) — lo usan SCGRec y CPGRec")
# gdown puede requerir reintento si Drive pide confirmación por tamaño/cuota
try:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "gdown"], check=False)
    import gdown, zipfile, tarfile, glob

    SCGREC_ID = "1F9kr_YWimBtexJEH-zkDzCOwl1q7GmFp"  # nota al pie 8 del paper SCGRec
    raw = gdown.download(id=SCGREC_ID, output="scgrec_raw", quiet=False)

    target = "scgrec_raw"
    if raw and zipfile.is_zipfile(raw):
        with zipfile.ZipFile(raw) as z:
            z.extractall("scgrec_data")
        target = "scgrec_data"
    elif raw and tarfile.is_tarfile(raw):
        with tarfile.open(raw) as t:
            t.extractall("scgrec_data")
        target = "scgrec_data"

    files = (glob.glob(os.path.join(target, "**", "*"), recursive=True)
             if os.path.isdir(target) else [target])
    print("\nArchivos SCGRec:")
    for f in sorted(files):
        if os.path.isfile(f):
            print(f"  {f}  ({os.path.getsize(f)/1e6:.1f} MB)")

    # Perfilado por tipo (estructura interna desconocida hasta correr)
    for f in sorted(files):
        if not os.path.isfile(f):
            continue
        low = f.lower()
        if low.endswith((".csv", ".tsv", ".txt")):
            try:
                df = pd.read_csv(f, sep=None, engine="python", nrows=200_000)
                profile_df(df, os.path.basename(f), id_cols=list(df.columns[:2]))
            except Exception as ee:
                print("  (no tabular legible:", os.path.basename(f), "->", repr(ee), ")")
        elif low.endswith((".json", ".jsonl")):
            with open(f, encoding="utf-8") as fh:
                line = fh.readline().strip()
            try:
                rec = json.loads(line)
            except Exception:
                rec = ast.literal_eval(line)
            print(f"  {os.path.basename(f)} · keys 1er registro:",
                  list(rec.keys()) if isinstance(rec, dict) else type(rec).__name__)
except Exception as e:
    print("‼ SCGRec falló:", repr(e))
    print("  (si es por cuota/confirmación de Drive, descarga manual desde el link de la nota 8 del paper)")


DATASET 2b · SCGRec (Yang et al. WWW'22) — lo usan SCGRec y CPGRec


Downloading...
From (original): https://drive.google.com/uc?id=1F9kr_YWimBtexJEH-zkDzCOwl1q7GmFp
From (redirected): https://drive.google.com/uc?id=1F9kr_YWimBtexJEH-zkDzCOwl1q7GmFp&confirm=t&uuid=51bee8bc-d42a-4d70-b5c8-75af92c0fa17
To: /content/scgrec_raw
100%|██████████| 993M/993M [00:16<00:00, 62.1MB/s]
/tmp/ipykernel_1382/2111861944.py:18: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall("scgrec_data")



Archivos SCGRec:
  scgrec_data/steam_data/App_ID_Info.txt  (0.1 MB)
  scgrec_data/steam_data/Games_Developers.txt  (0.1 MB)
  scgrec_data/steam_data/Games_Genres.txt  (0.1 MB)
  scgrec_data/steam_data/Games_Publishers.txt  (0.1 MB)
  scgrec_data/steam_data/Groups.txt  (272.3 MB)
  scgrec_data/steam_data/app_id.txt  (0.0 MB)
  scgrec_data/steam_data/friends.txt  (595.0 MB)
  scgrec_data/steam_data/test_data/test_game.txt  (1.5 MB)
  scgrec_data/steam_data/test_data/test_time.txt  (1.3 MB)
  scgrec_data/steam_data/train_game.txt  (585.6 MB)
  scgrec_data/steam_data/train_time.txt  (402.1 MB)
  scgrec_data/steam_data/user_game.txt  (2566.9 MB)
  scgrec_data/steam_data/users.txt  (70.4 MB)
  scgrec_data/steam_data/valid_data/valid_game.txt  (1.5 MB)
  scgrec_data/steam_data/valid_data/valid_time.txt  (1.3 MB)

--- App_ID_Info.txt ---
shape: 2,237 filas x 8 columnas
                       dtype  n_missing  %_missing  n_unique
10                     int64          0        0.0      2237
Cou

## AMAZON Video Games 2023 (opcional, cross-domain)

In [6]:
banner("DATASET 3 · AMAZON Video Games 2023 (opcional)")
AMZ = {
    "meta":   "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Video_Games.jsonl.gz",
    "review": "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Video_Games.jsonl.gz",
}


def peek_jsonl_gz(url, n=3):
    """Lee solo las primeras n líneas por streaming (memory-safe)."""
    print("leyendo", url)
    req = urllib.request.Request(url, headers=HEADERS)
    out = []
    with urllib.request.urlopen(req) as resp, gzip.GzipFile(fileobj=resp) as gz:
        for i, line in enumerate(gz):
            out.append(json.loads(line))
            if i + 1 >= n:
                break
    return out


try:
    meta = peek_jsonl_gz(AMZ["meta"])
    print(">>> columnas META Video_Games:", list(meta[0].keys()))
    print("ejemplo categories:", meta[0].get("categories"))
    rev = peek_jsonl_gz(AMZ["review"])
    print(">>> columnas REVIEW Video_Games:", list(rev[0].keys()))
    print("(conteos exactos por categoría: ver la tabla oficial de Amazon-Reviews-2023)")
except Exception as e:
    print("‼ Amazon falló:", repr(e))


DATASET 3 · AMAZON Video Games 2023 (opcional)
leyendo https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Video_Games.jsonl.gz
>>> columnas META Video_Games: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together']
ejemplo categories: ['Video Games', 'PC', 'Games']
leyendo https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Video_Games.jsonl.gz
>>> columnas REVIEW Video_Games: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']
(conteos exactos por categoría: ver la tabla oficial de Amazon-Reviews-2023)


## FIN

In [7]:
banner("LISTO — copia toda la salida y pégala de vuelta para actualizar DATASETS.md")


LISTO — copia toda la salida y pégala de vuelta para actualizar DATASETS.md
